# 02 - Beacon Candidate Detection

This notebook explores Phase 3 candidate detection. The detector finds all possible bright objects; it does not decide which one is the true terminal. Correct identification is the CNN phase.

In [ ]:
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import PreprocessingConfig, load_image
from src.candidate_detector import DetectorConfig, detect_candidates, draw_candidates

IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "python-generated"
LABELS_PATH = PROJECT_ROOT / "data" / "labels" / "labels.csv"
CANDIDATES_PATH = PROJECT_ROOT / "outputs" / "candidate-detection" / "candidates.csv"
SUMMARY_PATH = PROJECT_ROOT / "outputs" / "candidate-detection" / "summary.json"

labels = pd.read_csv(LABELS_PATH)
candidates = pd.read_csv(CANDIDATES_PATH)
candidates.head()

## Detector Configuration Used For Final Run

In [ ]:
preprocessing_config = PreprocessingConfig(
    threshold_method="fixed",
    threshold_value=180,
    blur_kernel=5,
    morph_kernel=3,
)
detector_config = DetectorConfig(
    min_area=3,
    max_area=1000,
    min_radius=1,
    max_radius=25,
    min_circularity=0.15,
    match_tolerance=12,
)
preprocessing_config, detector_config

## Candidate Counts By Scenario

In [ ]:
frame_counts = candidates.groupby("filename").size().rename("candidate_count").reset_index()
merged = labels.merge(frame_counts, on="filename", how="left").fillna({"candidate_count": 0})
merged["candidate_count"] = merged["candidate_count"].astype(int)

summary = merged.groupby("scenario_type")["candidate_count"].agg(["count", "mean", "min", "max"])
summary

## Verify Multiple-Beacon Frames

Multiple-beacon frames should often have more than one candidate, because false beacons are valid candidates even though they are not the true target.

In [ ]:
multi = merged[merged["scenario_type"] == "target_with_false_beacons"]
print("total multiple-beacon frames:", len(multi))
print("frames with 2+ candidates:", int((multi["candidate_count"] >= 2).sum()))
print("max candidates:", int(multi["candidate_count"].max()))
multi[["filename", "decoy_count", "candidate_count"]].head(10)

## Run Detector On Representative Samples

In [ ]:
samples = labels.groupby("scenario_id", sort=True).head(1).reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, row in zip(axes.flatten(), samples.itertuples(index=False)):
    frame = load_image(IMAGE_DIR / row.filename)
    result = detect_candidates(frame, preprocessing_config, detector_config)
    gt = (int(row.target_x), int(row.target_y), int(row.target_radius)) if row.target_visible else None
    overlay = draw_candidates(frame, result["candidates"], result["best_candidate"], ground_truth=gt)
    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(f"S{row.scenario_id}: {row.scenario_type}")
    ax.axis("off")
plt.tight_layout()

## Candidate Recall Metrics

The key metric for candidate detection is recall: the real beacon must appear somewhere in the candidate list. Baseline top-1 is useful, but the CNN classifier is responsible for final identity selection.

In [ ]:
import json

with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    detector_summary = json.load(f)

detector_summary["evaluation"]

## Useful Output Files

- Candidate rows: `outputs/candidate-detection/candidates.csv`
- Metrics summary: `outputs/candidate-detection/summary.json`
- Overlays: `outputs/candidate-detection/overlays/`
- Overlay montage: `outputs/candidate-detection/overlay_montage_20.png`